In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.collections import PatchCollection
import numpy as np
from matplotlib import patheffects
import matplotlib.gridspec as gridspec

# Palette
BG = '#0B1120'
DEEP_BLUE = '#065A82'
TEAL = '#1C7293'
MIDNIGHT = '#21295C'
ACCENT = '#E74C3C'
LIGHT = '#C8D6E5'
WHITE = '#FFFFFF'
MUTED = '#5A6C7D'
WARM = '#F39C12'
GREEN = '#27AE60'
PURPLE = '#8E44AD'

SAVE_DIR = '/Users/sol/Documents/Obsidian Vault/slides-assets'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor': BG,
    'text.color': WHITE,
    'axes.labelcolor': LIGHT,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 13,
})
print('Setup done')

# 1. Score Progression — Ridgeline with Glow
The heartbeat of 48 experiments. Filled area under the running best, glowing edge, failures as faint scars below.

In [ ]:
experiments = [
    (1, 0.874638, 'keep', 1), (2, 0.876672, 'keep', 1), (3, 0.897816, 'keep', 1),
    (4, 0.821483, 'discard', 1), (5, 0.891700, 'discard', 1), (6, 0.889260, 'discard', 1),
    (7, 0.914849, 'keep', 1), (8, 0.917557, 'keep', 1), (9, 0.855421, 'discard', 1),
    (10, 0.837744, 'discard', 1), (11, 0.924063, 'keep', 1), (12, 0.863828, 'discard', 1),
    (13, 0.929441, 'keep', 1), (14, 0.883816, 'discard', 1), (15, 0.926728, 'discard', 1),
    (16, 0.929267, 'discard', 1), (17, 0.913348, 'discard', 1), (18, 0.850535, 'discard', 1),
    (19, 0.923974, 'discard', 1), (20, 0.885704, 'discard', 1),
    (21, 0.918324, 'discard', 2), (22, 0.926428, 'discard', 2), (23, 0.916602, 'discard', 2),
    (24, 0.908340, 'discard', 2), (25, 0.927440, 'discard', 1), (26, 0.919782, 'discard', 1),
    (27, 0.000000, 'crash', 2), (28, 0.849953, 'discard', 2), (29, 0.865757, 'discard', 2),
    (30, 0.904438, 'discard', 2), (31, 0.798865, 'discard', 1), (32, 0.929441, 'discard', 1),
    (33, 0.929441, 'keep', 2), (34, 0.375506, 'discard', 2), (35, 0.923627, 'discard', 2),
    (36, 0.929441, 'keep', 3), (37, 0.929441, 'keep', 3), (38, 0.929441, 'discard', 3),
    (39, 0.929715, 'keep', 3), (40, 0.929441, 'discard', 3),
    (41, 0.938439, 'keep', 3), (42, 0.958851, 'keep', 3), (43, 0.961615, 'keep', 3),
    (44, 0.962205, 'keep', 3), (45, 0.966452, 'keep', 3), (46, 0.962161, 'discard', 3),
    (47, 0.969224, 'keep', 3),
]

fig, ax = plt.subplots(figsize=(16, 6))

# Running best
best_x, best_y = [], []
current_best = 0
for idx, score, status, tier in experiments:
    if status != 'crash' and score > current_best:
        current_best = score
    best_x.append(idx)
    best_y.append(current_best)

# Filled area under running best — gradient effect via layered fills
base = 0.74
for alpha, offset in [(0.03, 0), (0.05, 0.005), (0.08, 0.01), (0.12, 0.015)]:
    adjusted = [max(base, y - offset) for y in best_y]
    ax.fill_between(best_x, base, adjusted, color=ACCENT, alpha=alpha)

# Glowing best line — multiple passes with decreasing width and increasing opacity
for width, alpha in [(8, 0.08), (5, 0.15), (3, 0.3), (1.5, 0.9)]:
    ax.step(best_x, best_y, where='post', color=ACCENT, linewidth=width, alpha=alpha)

# Vertical drop lines from failures — "scars"
for idx, score, status, tier in experiments:
    if status == 'crash':
        continue
    best_at = best_y[idx - 1]
    if status == 'discard' and score < best_at - 0.01:
        ax.plot([idx, idx], [best_at, score], color=MUTED, alpha=0.15, linewidth=0.8)
        ax.scatter(idx, score, color=MUTED, s=12, alpha=0.25, marker='v', zorder=3)

# Kept experiments — bright dots on the ridge
for idx, score, status, tier in experiments:
    if status == 'keep':
        glow_color = {1: TEAL, 2: WARM, 3: GREEN}[tier]
        ax.scatter(idx, score, color=glow_color, s=60, zorder=6,
                   edgecolors=WHITE, linewidths=0.8, alpha=0.95)

# Tier zones — subtle vertical bands
ax.axvspan(0.5, 20.5, alpha=0.04, color=TEAL)
ax.axvspan(20.5, 35.5, alpha=0.04, color=WARM)
ax.axvspan(35.5, 47.5, alpha=0.04, color=GREEN)

# Tier labels at top
for label, x, color in [('HANDCRAFTED', 10.5, TEAL), ('PRETRAINED', 28, WARM), ('SELF-TAUGHT CNN', 41.5, GREEN)]:
    ax.text(x, 0.99, label, ha='center', fontsize=10, color=color, alpha=0.6,
            fontweight='bold', fontfamily='monospace', letterspacing=0.15)

# Key moment annotations with custom styling
ax.annotate('', xy=(13, 0.929), xytext=(13, 0.975),
            arrowprops=dict(arrowstyle='-', color=TEAL, lw=0.8, linestyle='--'))
ax.text(13, 0.978, 'NDSI\n0.929', ha='center', fontsize=9, color=TEAL, fontweight='bold')

# The breakthrough moment — experiment 42
ax.annotate('', xy=(42, 0.959), xytext=(42, 0.985),
            arrowprops=dict(arrowstyle='-', color=GREEN, lw=1.2, linestyle='--'))
ax.text(42, 0.988, 'CNN 100ep\n+0.030 jump', ha='center', fontsize=9, color=GREEN, fontweight='bold')

# Final score callout
ax.text(47.8, 0.969, '0.969', fontsize=20, fontweight='bold', color=ACCENT,
        ha='left', va='center',
        path_effects=[patheffects.withStroke(linewidth=3, foreground=BG)])

# The dead zone annotation
ax.text(28, 0.77, 'every pretrained model failed', ha='center', fontsize=9,
        color=WARM, alpha=0.5, style='italic')

ax.set_xlim(0, 49)
ax.set_ylim(0.74, 1.0)
ax.set_xlabel('Experiment', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(MUTED)
ax.spines['bottom'].set_color(MUTED)
ax.set_yticks([0.80, 0.85, 0.90, 0.95, 1.0])
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.2f'))
ax.grid(axis='y', color=MUTED, alpha=0.08)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/score_progression.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved score_progression.png')

# 2. Day/Night — 24-Hour Polar Clock
Each ring is an environment type. Segments light up at the hours they were detected. Shows the circadian pattern from pure audio.

In [ ]:
# Approximate hour distributions from cluster analysis
# Vessels: 6am-6pm (100% daytime)
# Silence: 6pm-6am mostly (98% nighttime)
# Biology: spread across 24h (50/50)
# Quiet ambient: daytime only

hours = np.arange(24)
theta = (hours / 24) * 2 * np.pi - np.pi/2  # 0h at top

# Intensity profiles (0-1) for each environment
vessel = np.array([0,0,0,0,0,0.3, 0.8,1,1,1,1,1, 1,1,1,0.9,0.7,0.3, 0,0,0,0,0,0])
silence = np.array([0.9,1,1,1,1,0.8, 0.1,0,0,0,0,0, 0,0,0,0,0.1,0.3, 0.7,0.9,1,1,1,0.95])
biology = np.array([0.5,0.5,0.4,0.4,0.5,0.5, 0.6,0.6,0.6,0.7,0.6,0.6, 0.5,0.5,0.6,0.6,0.5,0.5, 0.5,0.4,0.5,0.5,0.5,0.5])

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
ax.set_facecolor(BG)
fig.patch.set_facecolor(BG)

bar_width = (2 * np.pi / 24) * 0.9

# Three concentric rings
ring_configs = [
    (vessel, 0.3, 0.7, ACCENT, 'Vessels (n=107)'),
    (silence, 0.75, 1.15, MIDNIGHT, 'Silence (n=343)'),
    (biology, 1.2, 1.6, TEAL, 'Reef biology (n=121)'),
]

for data, r_inner, r_outer, color, label in ring_configs:
    heights = data * (r_outer - r_inner)
    for i in range(24):
        alpha = 0.15 + 0.85 * data[i]
        ax.bar(theta[i], heights[i], width=bar_width, bottom=r_inner,
               color=color, alpha=alpha, edgecolor=BG, linewidth=0.5)
    # Label
    mid_r = (r_inner + r_outer) / 2
    ax.text(np.pi * 0.75, mid_r + 0.15, label, ha='center', va='center',
            fontsize=9, color=color, fontweight='bold', alpha=0.8)

# Hour labels
for h in [0, 6, 12, 18]:
    angle = (h / 24) * 2 * np.pi - np.pi/2
    labels = {0: '00:00', 6: '06:00', 12: '12:00', 18: '18:00'}
    ax.text(angle, 1.85, labels[h], ha='center', va='center',
            fontsize=10, color=LIGHT, fontweight='bold')

# Day/night shading
day_theta = np.linspace(-np.pi/2 + (6/24)*2*np.pi, -np.pi/2 + (18/24)*2*np.pi, 100)
night_theta = np.concatenate([
    np.linspace(-np.pi/2 + (18/24)*2*np.pi, -np.pi/2 + 2*np.pi, 50),
    np.linspace(-np.pi/2, -np.pi/2 + (6/24)*2*np.pi, 50)
])

# Subtle day/night indicator at outer edge
for t in day_theta:
    ax.bar(t, 0.06, width=0.03, bottom=1.7, color=WARM, alpha=0.3)
for t in night_theta:
    ax.bar(t, 0.06, width=0.03, bottom=1.7, color=MIDNIGHT, alpha=0.5)

ax.text(-np.pi/2 + np.pi/2, 1.95, 'DAY', fontsize=8, color=WARM, alpha=0.5,
        ha='center', va='center')
ax.text(-np.pi/2 + np.pi*1.5, 1.95, 'NIGHT', fontsize=8, color=LIGHT, alpha=0.3,
        ha='center', va='center')

# Center text
ax.text(0, 0, 'zero\ntemporal\nfeatures', ha='center', va='center',
        fontsize=11, color=MUTED, style='italic')

ax.set_ylim(0, 2.1)
ax.set_xticks([])
ax.set_yticks([])
ax.spines['polar'].set_visible(False)
ax.grid(False)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/day_night.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved day_night.png')

# 3. Band Profiles — Frequency Heatmap
Rows = environments, columns = frequency bands. Color = power. The one bright cell in mid-freq for biology tells the whole story.

In [ ]:
# Power in dB (higher = louder), normalized to 0-1 for visualization
# Original dB: Vessel loud [-62, -120, -120], Vessel med [-76, -117, -120],
#              Silence [-117, -120, -120], Biology [-82, -82, -95], Quiet [-91, -103, -105]

envs = ['Vessel\n(loud)', 'Vessel\n(medium)', 'Reef\nbiology', 'Quiet\nambient', 'Deep\nsilence']
bands = ['LOW\n50Hz–2kHz', 'MID\n2–20kHz', 'HIGH\n20–24kHz']

# Normalize: -120 dB = 0 (silence), -60 dB = 1 (loudest)
raw = np.array([
    [-61.9, -119.8, -120.0],   # Vessel loud
    [-76.1, -117.0, -119.9],   # Vessel medium
    [-81.7, -81.7,  -95.1],    # Biology
    [-90.7, -102.6, -104.8],   # Quiet
    [-117.4, -119.9, -120.0],  # Silence
])
norm = (raw - (-120)) / ((-60) - (-120))  # 0=silent, 1=loud
norm = np.clip(norm, 0, 1)

fig, ax = plt.subplots(figsize=(8, 6))

# Custom colormap: dark blue (silent) → teal → bright white (loud)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ocean_heat',
    [(0, '#0B1120'), (0.2, '#0D1B2A'), (0.4, '#1C7293'), (0.6, '#27AE60'), (0.8, '#F39C12'), (1.0, '#E74C3C')])

im = ax.imshow(norm, cmap=cmap, aspect='auto', vmin=0, vmax=1)

# Cell annotations
for i in range(len(envs)):
    for j in range(len(bands)):
        val = norm[i, j]
        db = raw[i, j]
        text_color = WHITE if val > 0.3 else MUTED
        weight = 'bold' if val > 0.4 else 'normal'
        ax.text(j, i, f'{db:.0f}\ndB', ha='center', va='center',
                fontsize=11, color=text_color, fontweight=weight)

# Highlight the biology mid-freq cell
rect = plt.Rectangle((0.5, 1.5), 1, 1, linewidth=2.5, edgecolor=WHITE, facecolor='none', linestyle='--')
ax.add_patch(rect)
ax.annotate('only cluster with\nmid-freq energy', xy=(1, 2), xytext=(2.6, 3.5),
            fontsize=10, color=WHITE, fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color=WHITE, lw=1.5))

ax.set_xticks(range(len(bands)))
ax.set_xticklabels(bands, fontsize=12, color=LIGHT)
ax.set_yticks(range(len(envs)))
ax.set_yticklabels(envs, fontsize=12, color=LIGHT)
ax.tick_params(length=0)

# Grid lines between cells
for i in range(len(envs) + 1):
    ax.axhline(i - 0.5, color=BG, linewidth=3)
for j in range(len(bands) + 1):
    ax.axvline(j - 0.5, color=BG, linewidth=3)

ax.set_xlim(-0.5, 2.5)
ax.set_ylim(4.5, -0.5)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/band_profiles.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved band_profiles.png')

# 4. Survival Grid — Waffle Chart of All 85 Experiments
Each cell = one experiment. Color = project. Fill = kept vs reverted. One image shows the natural selection across all three problems.

In [ ]:
# All experiments across 3 projects
# Acoustics: 14 kept, 33 reverted, 1 crashed = 48
# BRUV: 8 kept, 12 reverted = 20
# Precip: 5 kept, 12 reverted = 17

# Build sequence: kept first then reverted for each project (shows survival)
cells = []
# Acoustics
for _ in range(14): cells.append(('Acoustics', 'kept'))
for _ in range(33): cells.append(('Acoustics', 'reverted'))
cells.append(('Acoustics', 'crashed'))
# BRUV
for _ in range(8): cells.append(('BRUV', 'kept'))
for _ in range(12): cells.append(('BRUV', 'reverted'))
# Precip
for _ in range(5): cells.append(('Precip', 'kept'))
for _ in range(12): cells.append(('Precip', 'reverted'))

# Colors: project → base color, kept = full, reverted = faded outline
project_colors = {'Acoustics': TEAL, 'BRUV': ACCENT, 'Precip': WARM}

fig, ax = plt.subplots(figsize=(14, 5))

cols = 17  # 85/5 = 17 columns
rows = 5
cell_size = 0.8
gap = 0.15

for i, (project, status) in enumerate(cells):
    row = i // cols
    col = i % cols
    x = col * (cell_size + gap)
    y = (rows - 1 - row) * (cell_size + gap)
    color = project_colors[project]
    
    if status == 'kept':
        rect = FancyBboxPatch((x, y), cell_size, cell_size, 
                               boxstyle='round,pad=0.05',
                               facecolor=color, edgecolor=color, linewidth=1.5, alpha=0.9)
    elif status == 'crashed':
        rect = FancyBboxPatch((x, y), cell_size, cell_size,
                               boxstyle='round,pad=0.05',
                               facecolor=BG, edgecolor=ACCENT, linewidth=1.5, linestyle='--', alpha=0.5)
        ax.text(x + cell_size/2, y + cell_size/2, 'X', ha='center', va='center',
                fontsize=8, color=ACCENT, fontweight='bold', alpha=0.6)
    else:
        rect = FancyBboxPatch((x, y), cell_size, cell_size,
                               boxstyle='round,pad=0.05',
                               facecolor=BG, edgecolor=color, linewidth=1, alpha=0.25)
    ax.add_patch(rect)

# Project divider lines and labels
# Acoustics: cells 0-47, BRUV: 48-67, Precip: 68-84
dividers = [(48, 'BRUV'), (68, 'Precip')]
for cell_idx, label in dividers:
    row = cell_idx // cols
    col = cell_idx % cols
    x = col * (cell_size + gap) - gap/2
    y_top = (rows - 1) * (cell_size + gap) + cell_size + 0.2
    y_bot = -0.3

# Labels above
ax.text(4 * (cell_size + gap), rows * (cell_size + gap) + 0.1,
        'ACOUSTICS (48)', ha='center', fontsize=11, color=TEAL, fontweight='bold')
ax.text(10.5 * (cell_size + gap), rows * (cell_size + gap) + 0.1,
        'BRUV (20)', ha='center', fontsize=11, color=ACCENT, fontweight='bold')
ax.text(14.5 * (cell_size + gap), rows * (cell_size + gap) + 0.1,
        'PRECIP (17)', ha='center', fontsize=11, color=WARM, fontweight='bold')

# Legend
legend_y = -0.8
ax.add_patch(FancyBboxPatch((0, legend_y), 0.5, 0.5, boxstyle='round,pad=0.03',
             facecolor=MUTED, edgecolor=MUTED, alpha=0.8))
ax.text(0.7, legend_y + 0.25, 'Kept (27)', va='center', fontsize=10, color=LIGHT)

ax.add_patch(FancyBboxPatch((3, legend_y), 0.5, 0.5, boxstyle='round,pad=0.03',
             facecolor=BG, edgecolor=MUTED, linewidth=1, alpha=0.4))
ax.text(3.7, legend_y + 0.25, 'Reverted (57)', va='center', fontsize=10, color=MUTED)

# Stats
ax.text(13 * (cell_size + gap), legend_y + 0.25,
        '85 experiments  ·  32% survival rate  ·  $33 compute',
        ha='right', fontsize=10, color=MUTED, style='italic')

ax.set_xlim(-0.3, cols * (cell_size + gap))
ax.set_ylim(legend_y - 0.3, rows * (cell_size + gap) + 0.5)
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/survival.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved survival.png')

# 5. BRUV PPF — Visual Equation
Show the cross-calibration as proportional pixel blocks. Sparse side: few big fish pixels. Dense side: massive pixel field. The ratio bridges them.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')

# === LEFT: Sparse scene ===
# Pixel grid representing 2479 pixels / 54 fish
# Show as a small grid of dots (each dot ≈ a fish's pixel footprint)
np.random.seed(42)
n_sparse = 54
sparse_x = np.random.uniform(0.3, 3.3, n_sparse)
sparse_y = np.random.uniform(1.0, 3.8, n_sparse)
ax.scatter(sparse_x, sparse_y, s=40, color=TEAL, alpha=0.7, edgecolors=WHITE, linewidths=0.3)

# Border
rect1 = FancyBboxPatch((0.1, 0.6), 3.4, 3.6, boxstyle='round,pad=0.1',
                         facecolor='none', edgecolor=TEAL, linewidth=2)
ax.add_patch(rect1)
ax.text(1.8, 4.5, 'SPARSE VIDEO', ha='center', fontsize=12, color=TEAL, fontweight='bold')
ax.text(1.8, 0.2, '54 fish  ·  2,479 px', ha='center', fontsize=10, color=LIGHT)

# === CENTER: The equation ===
# Arrow from sparse
ax.annotate('', xy=(5.2, 2.5), xytext=(3.7, 2.5),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))

# PPF box
ppf_box = FancyBboxPatch((5.2, 1.5), 3.0, 2.0, boxstyle='round,pad=0.15',
                          facecolor=MIDNIGHT, edgecolor=ACCENT, linewidth=2.5)
ax.add_patch(ppf_box)
ax.text(6.7, 3.1, 'PIXELS PER FISH', ha='center', fontsize=9, color=LIGHT,
        fontfamily='monospace', fontweight='bold')
ax.text(6.7, 2.3, '2479 ÷ 54', ha='center', fontsize=14, color=MUTED)
ax.text(6.7, 1.7, '= 45.9', ha='center', fontsize=22, color=ACCENT, fontweight='bold')

# Arrow to dense
ax.annotate('', xy=(10.0, 2.5), xytext=(8.4, 2.5),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))

# === RIGHT: Dense scene ===
# Many many dots packed tight
n_dense = 251
dense_x = np.random.uniform(10.1, 13.5, n_dense)
dense_y = np.random.uniform(1.0, 3.8, n_dense)
ax.scatter(dense_x, dense_y, s=15, color=ACCENT, alpha=0.6, edgecolors=WHITE, linewidths=0.1)

rect2 = FancyBboxPatch((9.9, 0.6), 3.8, 3.6, boxstyle='round,pad=0.1',
                         facecolor='none', edgecolor=ACCENT, linewidth=2)
ax.add_patch(rect2)
ax.text(11.8, 4.5, 'DENSE VIDEO', ha='center', fontsize=12, color=ACCENT, fontweight='bold')
ax.text(11.8, 0.2, '11,509 px ÷ 45.9 = 251', ha='center', fontsize=10, color=LIGHT, fontweight='bold')

# Bottom tagline
ax.text(7.0, -0.1, 'sparse calibrates dense', ha='center', fontsize=11,
        color=MUTED, style='italic')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/bruv_ppf.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved bruv_ppf.png')

# 6. Precip Imbalance — Scaled Area Blocks
No donut. Show the 87/12/1 split as proportional AREA. 87 large blocks, 12 medium blocks, 1 tiny block.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.axis('off')

# Treemap-style: areas proportional to class frequency
# No rain: 87% → big block
# Light rain: 12% → medium block
# Heavy rain: 1% → tiny block

# No rain block
nr = FancyBboxPatch((0.2, 0.5), 7.5, 4.0, boxstyle='round,pad=0.1',
                     facecolor=MIDNIGHT, edgecolor=MUTED, linewidth=1.5, alpha=0.6)
ax.add_patch(nr)
ax.text(4.0, 2.8, '87%', ha='center', fontsize=48, color=MUTED, fontweight='bold', alpha=0.4)
ax.text(4.0, 1.6, 'no rain', ha='center', fontsize=16, color=MUTED, alpha=0.6)

# Light rain block
lr = FancyBboxPatch((8.0, 1.7), 2.8, 2.8, boxstyle='round,pad=0.1',
                     facecolor=TEAL, edgecolor=TEAL, linewidth=1.5, alpha=0.4)
ax.add_patch(lr)
ax.text(9.4, 3.3, '12%', ha='center', fontsize=24, color=TEAL, fontweight='bold')
ax.text(9.4, 2.4, 'light rain', ha='center', fontsize=11, color=TEAL)

# Heavy rain — tiny!
hr = FancyBboxPatch((8.0, 0.5), 0.9, 0.9, boxstyle='round,pad=0.05',
                     facecolor=ACCENT, edgecolor=ACCENT, linewidth=2, alpha=0.8)
ax.add_patch(hr)
ax.text(8.45, 0.95, '1%', ha='center', fontsize=12, color=WHITE, fontweight='bold')

# Arrow pointing to the tiny block
ax.annotate('heavy rain — the signal\nthe agent most needs\nis almost never there',
            xy=(8.45, 0.5), xytext=(10.5, 0.7),
            fontsize=10, color=ACCENT, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5))

# Heavy rain F1 collapse
ax.text(0.5, 0.15, 'Heavy rain F1:  3h → 0.35    6h → 0.18    12h → 0.02',
        fontsize=10, color=MUTED, style='italic')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/precip_bottleneck.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved precip_bottleneck.png')

# 7. Infrastructure — Layered Network Diagram
Three layers: Agents (top), Compute (middle), Data (bottom). Clean cards with subtle rounded corners.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7.5)
ax.axis('off')

def card(ax, x, y, w, h, title, subtitle, detail, score, color, border_color):
    """Draw a styled card with accent stripe on left"""
    # Main card
    main = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                           facecolor='#0D1B2A', edgecolor=border_color, linewidth=1.5)
    ax.add_patch(main)
    # Left accent stripe
    stripe = plt.Rectangle((x + 0.05, y + 0.15), 0.08, h - 0.3,
                            facecolor=color, alpha=0.8)
    ax.add_patch(stripe)
    # Text
    ax.text(x + 0.35, y + h - 0.35, title, fontsize=11, color=WHITE, fontweight='bold')
    ax.text(x + 0.35, y + h - 0.7, subtitle, fontsize=9, color=color)
    ax.text(x + 0.35, y + h - 1.0, detail, fontsize=7.5, color=MUTED)
    if score:
        ax.text(x + w - 0.2, y + 0.25, score, fontsize=14, color=color,
                fontweight='bold', ha='right')

# === AGENT LAYER (top) ===
ax.text(7, 7.2, 'AGENTS', ha='center', fontsize=9, color=MUTED,
        fontfamily='monospace', fontweight='bold', alpha=0.5)

card(ax, 0.3, 5.5, 3.8, 1.5, 'Marine Acoustics', 'CNN + UMAP + HDBSCAN',
     '48 experiments · 29% survival', '0.979', TEAL, TEAL)
card(ax, 5.1, 5.5, 3.8, 1.5, 'BRUV Fish Counting', 'BG sub + YOLO + IoU tracking',
     '20 experiments · 40% survival', '0.998', ACCENT, ACCENT)
card(ax, 9.9, 5.5, 3.8, 1.5, 'Precipitation', 'Cascade RF + LGB blend',
     '17 experiments · 29% survival', '0.875', WARM, WARM)

# === COMPUTE LAYER (middle) ===
ax.text(7, 5.15, 'COMPUTE', ha='center', fontsize=9, color=MUTED,
        fontfamily='monospace', fontweight='bold', alpha=0.5)

# Compute nodes
for cx, label, specs, col in [
    (2.2, 'MacBook Pro', 'Apple M-series · MPS', TEAL),
    (7, 'GPU #1 — g4dn.xlarge', 'Shared: Marine + Precip', WARM),
    (11.8, 'GPU #2 — g4dn.2xlarge', 'Dedicated: BRUV', ACCENT),
]:
    box = FancyBboxPatch((cx - 1.7, 3.6), 3.4, 1.2, boxstyle='round,pad=0.08',
                          facecolor=MIDNIGHT, edgecolor=col, linewidth=1, alpha=0.5)
    ax.add_patch(box)
    ax.text(cx, 4.4, label, ha='center', fontsize=9, color=LIGHT, fontweight='bold')
    ax.text(cx, 4.0, specs, ha='center', fontsize=7.5, color=MUTED)

# TALK.md coordination badge
talk_box = FancyBboxPatch((5.5, 3.1), 3.0, 0.4, boxstyle='round,pad=0.05',
                           facecolor=BG, edgecolor=MUTED, linewidth=1, linestyle='--')
ax.add_patch(talk_box)
ax.text(7, 3.3, 'TALK.md  ←  git branch message bus', ha='center',
        fontsize=8, color=MUTED, style='italic')

# === DATA LAYER (bottom) ===
ax.text(7, 2.7, 'DATA', ha='center', fontsize=9, color=MUTED,
        fontfamily='monospace', fontweight='bold', alpha=0.5)

for dx, label, items in [
    (3.5, 'Cloudflare R2', '11 WAVs · 18 videos (65GB) · 4 CSVs'),
    (10.5, 'AWS S3', 'Code snapshots · Metrics · Annotated video'),
]:
    box = FancyBboxPatch((dx - 2.5, 1.2), 5.0, 1.2, boxstyle='round,pad=0.08',
                          facecolor='#060D18', edgecolor=MUTED, linewidth=1,
                          linestyle='--', alpha=0.5)
    ax.add_patch(box)
    ax.text(dx, 2.0, label, ha='center', fontsize=10, color=LIGHT, fontweight='bold')
    ax.text(dx, 1.55, items, ha='center', fontsize=8, color=MUTED)

# Connection lines
for x_from, x_to in [(2.2, 2.2), (7, 7), (11.8, 11.8)]:
    ax.plot([x_from, x_to], [5.5, 4.8], color=MUTED, linewidth=0.8, alpha=0.3, linestyle='--')

# Dashed lines from compute to data
for x_from, x_to in [(2.2, 3.5), (7, 3.5), (7, 10.5), (11.8, 10.5)]:
    ax.plot([x_from, x_to], [3.6, 2.4], color=MUTED, linewidth=0.6, alpha=0.2, linestyle=':')

# Bottom stats
ax.text(7, 0.6, '~$33 total  ·  48 hours  ·  8 vCPU quota  ·  3 concurrent agents',
        ha='center', fontsize=10, color=MUTED)
ax.text(7, 0.2, 'Agents launched and terminated their own EC2 instances',
        ha='center', fontsize=9, color=MUTED, style='italic', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/infra_diagram.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved infra_diagram.png')

# 8. BRUV Progression — Connected Journey Path
Not bars. A winding path from left to right, with nodes at each milestone. Height = score. Width of path varies with confidence.

In [ ]:
milestones = [
    ('Area density', 0.325, 1),
    ('Pixel density', 0.748, 1),
    ('Dual BG', 0.785, 1),
    ('Multi-tier', 0.82, 2),
    ('Bait mask', 0.85, 2),
    ('YOLO PPF', 0.90, 2),
    ('Harmonic agg', 0.95, 2),
    ('Cross-vid PPF', 0.97, 2),
    ('Two-pass', 0.990, 3),
    ('Track-PPF', 0.998, 3),
]

fig, ax = plt.subplots(figsize=(14, 5))

xs = np.linspace(1, 13, len(milestones))
ys = [m[1] for m in milestones]
tier_colors_map = {1: TEAL, 2: DEEP_BLUE, 3: GREEN}

# Smooth interpolated path
from scipy.interpolate import make_interp_spline
x_smooth = np.linspace(xs[0], xs[-1], 200)
try:
    spl = make_interp_spline(xs, ys, k=3)
    y_smooth = spl(x_smooth)
except:
    y_smooth = np.interp(x_smooth, xs, ys)

# Gradient fill under curve
for i in range(len(x_smooth) - 1):
    progress = i / len(x_smooth)
    color = TEAL if progress < 0.3 else (DEEP_BLUE if progress < 0.7 else GREEN)
    ax.fill_between(x_smooth[i:i+2], 0.2, y_smooth[i:i+2], color=color, alpha=0.06)

# Glowing path line
for width, alpha in [(6, 0.08), (3, 0.2), (1.5, 0.8)]:
    ax.plot(x_smooth, y_smooth, color=WHITE, linewidth=width, alpha=alpha)

# Milestone nodes
for i, (label, score, tier) in enumerate(milestones):
    color = tier_colors_map[tier]
    size = 120 if i == len(milestones) - 1 else 60
    ax.scatter(xs[i], score, s=size, color=color, zorder=5,
               edgecolors=WHITE, linewidths=1.5)
    
    # Alternating label positions
    offset = 0.06 if i % 2 == 0 else -0.06
    va = 'bottom' if i % 2 == 0 else 'top'
    ax.text(xs[i], score + offset, f'{label}\n{score:.3f}',
            ha='center', va=va, fontsize=8, color=LIGHT,
            fontweight='bold' if i == len(milestones)-1 else 'normal')

# Final score highlight
ax.text(13.5, 0.998, '0.998', fontsize=22, fontweight='bold', color=ACCENT,
        ha='left', va='center',
        path_effects=[patheffects.withStroke(linewidth=3, foreground=BG)])

# Tier brackets at bottom
for label, x_start, x_end, color in [
    ('Classical CV', xs[0], xs[2], TEAL),
    ('YOLO + Calibration', xs[3], xs[7], DEEP_BLUE),
    ('Tracking PPF', xs[8], xs[9], GREEN),
]:
    mid = (x_start + x_end) / 2
    ax.plot([x_start, x_end], [0.25, 0.25], color=color, linewidth=2, alpha=0.4)
    ax.text(mid, 0.22, label, ha='center', fontsize=8, color=color, alpha=0.6)

ax.set_ylim(0.15, 1.1)
ax.set_xlim(0.3, 14.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(MUTED)
ax.spines['bottom'].set_visible(False)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.1f'))
ax.set_xticks([])
ax.grid(axis='y', color=MUTED, alpha=0.08)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/bruv_progression.png', dpi=200, bbox_inches='tight',
            facecolor=BG, edgecolor='none')
plt.close()
print('Saved bruv_progression.png')